**⭐ 1. What This Pattern Solves**

Detects changes in incoming data schema that could break ETL pipelines or downstream analytics.
Handles dynamic sources where fields may be added, removed, or type-changed.
Alerts or triggers transformations to adapt automatically, preventing runtime failures.
Useful in data lakes, streaming ingestion, and automated reporting pipelines.

**⭐ 2. SQL Equivalent**

In [0]:
%sql
-- Compare expected vs actual columns in a table
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'incoming_data'
EXCEPT
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'expected_schema';

**⭐ 3. Core Idea**

Track expected schema → Compare with current data → Flag or adapt changes.
Works because discrepancies can be programmatically detected before processing.

**⭐ 4. Template Code (MEMORIZE THIS)**

In [0]:
# expected_schema: dict of column -> type
# df: incoming DataFrame (pandas or PySpark)
def detect_schema_drift(df, expected_schema):
    current_schema = {col: str(df[col].dtype) for col in df.columns}
    
    added = set(current_schema) - set(expected_schema)
    removed = set(expected_schema) - set(current_schema)
    type_changed = {col for col in set(current_schema) & set(expected_schema)
                    if current_schema[col] != expected_schema[col]}
    
    drift = {
        "added_columns": list(added),
        "removed_columns": list(removed),
        "type_changes": list(type_changed)
    }
    return drift

**⭐ 5. Detailed Example**

In [0]:
expected_schema = {"id": "int64", "name": "object", "age": "int64"}

import pandas as pd
df = pd.DataFrame({
    "id": [1,2],
    "name": ["Alice", "Bob"],
    "salary": [50000, 60000]  # New column
})

drift = detect_schema_drift(df, expected_schema)
print(drift)

{
  'added_columns': ['salary'],
  'removed_columns': ['age'],
  'type_changes': []
}

**⭐ 6. Mini Practice Problems**

Incoming JSON data sometimes contains a timestamp field in string format; detect if the type changes to datetime.
Detect schema drift when a new optional column email is added in a streaming CSV source.
Compare two nested dictionaries representing expected vs actual JSON schemas.

**⭐ 7. Full Data Engineering Scenario**

Problem Statement:
A financial transactions pipeline ingests CSVs from multiple banks. Occasionally, banks add/remove columns or change types. You must detect schema drift and log changes before ETL.

Expected Output:

Added/removed columns list
Columns with type changes
Optionally, alert or auto-adjust ETL

Skeleton Solution:

In [0]:
# Step 1: Load expected schema
expected_schema = load_schema("expected_schema.json")

# Step 2: Load new batch
df = load_csv("bank_batch.csv")

# Step 3: Detect drift
drift = detect_schema_drift(df, expected_schema)

# Step 4: Alert or log
if any(drift.values()):
    log_drift(drift)
    trigger_etl_adjustments(drift)

**⭐ 8. Time & Space Complexity**

Time Complexity: O(n) where n = number of columns (comparison sets & type checks)
Space Complexity: O(n) for storing current schema & drift results

**⭐ 9. Common Pitfalls & Mistakes**

❌ Mistake: Only checking column names, ignoring type changes
❌ Mistake: Not handling nullable or optional fields, causing false positives
❌ Mistake: Comparing types as Python objects instead of strings, leading to mismatches
✔ Correct Approach: Compare both column presence and data types, log drift with detail
✔ Optimization: Use sets for column comparison for O(1) membership checks
✔ Production Tip: Integrate with alerting or automated schema evolution mechanisms